# RescueVision Edge — Training Notebook
**Hackathon FindIT! 2026 — Track A: The Edge Vision**

Model: YOLOv8n (Ultralytics) → ONNX export  
Dataset: VisDrone-DET 2019, kelas pedestrian only  

> **PENTING:** Jalankan semua cell secara berurutan. Jangan clear output sebelum submit — log training adalah bukti constraint compliance.

## 0. Environment Check

In [ ]:
import sys, os, platform
import torch

print(f'Python     : {sys.version}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')
print(f'Platform   : {platform.system()} {platform.machine()}')
print(f'Working dir: {os.getcwd()}')

# Pastikan working dir adalah root repo
if not os.path.exists('dataset.yaml'):
    os.chdir('..')  # naik satu level jika dijalankan dari notebooks/
    print(f'Changed to : {os.getcwd()}')

In [ ]:
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

# Cek dataset tersedia
from pathlib import Path
assert Path('train_data/images/train').exists(), 'STOP: Jalankan src/prepare_visdrone.py dulu'
assert Path('test_data/images').exists(), 'STOP: test_data belum ada'

train_imgs = list(Path('train_data/images/train').glob('*.jpg'))
val_imgs   = list(Path('train_data/images/val').glob('*.jpg'))
test_imgs  = list(Path('test_data/images').glob('*.jpg'))
print(f'Train images: {len(train_imgs)}')
print(f'Val images  : {len(val_imgs)}')
print(f'Test images : {len(test_imgs)}')

## 1. Dataset Verification (Data Leakage Check)

In [ ]:
# WAJIB: Buktikan tidak ada overlap antara train dan test
train_names = set(p.stem for p in Path('train_data/images/train').glob('*.jpg'))
val_names   = set(p.stem for p in Path('train_data/images/val').glob('*.jpg'))
test_names  = set(p.stem for p in Path('test_data/images').glob('*.jpg'))

overlap_train_test = train_names & test_names
overlap_val_test   = val_names & test_names
overlap_train_val  = train_names & val_names

print('=== Data Leakage Validation ===')
print(f'Train ∩ Test : {len(overlap_train_test)} files  ← harus 0')
print(f'Val   ∩ Test : {len(overlap_val_test)} files  ← harus 0')
print(f'Train ∩ Val  : {len(overlap_train_val)} files  ← harus 0')

assert len(overlap_train_test) == 0, f'DATA LEAKAGE DETECTED: {overlap_train_test}'
assert len(overlap_val_test)   == 0, f'DATA LEAKAGE DETECTED: {overlap_val_test}'
assert len(overlap_train_val)  == 0, f'DATA LEAKAGE DETECTED: {overlap_train_val}'
print('✅ No data leakage detected — constraint compliance OK')

## 2. Training Configuration

In [ ]:
# Konfigurasi training — tuning untuk VisDrone small object
TRAIN_CONFIG = {
    'data'       : 'dataset.yaml',
    'epochs'     : 100,           # Minimum. Naikkan ke 150+ jika waktu cukup
    'imgsz'      : 640,           # Input resolution. VisDrone butuh resolusi tinggi
    'batch'      : 16,            # Sesuaikan dengan VRAM GPU lab
    'device'     : 0,             # GPU index. Ganti ke 'cpu' jika tidak ada GPU
    'workers'    : 4,
    'patience'   : 20,            # Early stopping
    'save'       : True,
    'save_period': 10,            # Save checkpoint setiap 10 epoch
    'project'    : 'runs/train',
    'name'       : 'rescuevision_v1',
    
    # Augmentasi — penting untuk generalisasi di kondisi bencana
    'hsv_h'      : 0.015,         # Hue jitter
    'hsv_s'      : 0.7,           # Saturation jitter (asap, debu)
    'hsv_v'      : 0.4,           # Value jitter (pencahayaan tidak merata)
    'flipud'     : 0.0,           # Vertical flip (drone view — jarang dibalik)
    'fliplr'     : 0.5,           # Horizontal flip
    'scale'      : 0.5,           # Scale jitter (variasi ketinggian drone)
    'mosaic'     : 1.0,           # Mosaic augmentation (efektif untuk small object)
    'mixup'      : 0.1,
    
    # Loss weights — prioritaskan recall untuk search & rescue
    'box'        : 7.5,
    'cls'        : 0.5,
    'dfl'        : 1.5,
}

print('Training config:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k:12s}: {v}')

## 3. Model Initialization

In [ ]:
# Load YOLOv8n pretrained (COCO) — fine-tune untuk VisDrone
model = YOLO('yolov8n.pt')  # Auto-download jika belum ada
print(model.info())

## 4. Training

In [ ]:
# ============================================================
# MULAI TRAINING
# Jangan interrupt kecuali terpaksa — log ini akan menjadi
# bukti training untuk penilaian juri
# ============================================================
results = model.train(**TRAIN_CONFIG)
print('\n=== Training Complete ===')
print(f'Best model saved at: {results.save_dir}/weights/best.pt')

## 5. Evaluation on Test Set

In [ ]:
import shutil

# Load best model
best_pt_path = f'runs/train/rescuevision_v1/weights/best.pt'
best_model = YOLO(best_pt_path)

# Evaluate on test set
metrics = best_model.val(
    data='dataset.yaml',
    split='test',
    imgsz=640,
    device='cpu',   # Validasi di CPU sesuai kondisi juri
    batch=1,
)

print('\n=== TEST SET METRICS ===')
print(f'mAP@50    : {metrics.box.map50:.4f}')
print(f'mAP@50-95 : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

# Copy ke folder model/
os.makedirs('model', exist_ok=True)
shutil.copy2(best_pt_path, 'model/best.pt')
print(f'\nBest weights copied to model/best.pt')

## 6. ONNX Export (Constraint C-A2, C-A4)

In [ ]:
import os

# Export ke ONNX — ini yang akan digunakan untuk inference di CPU
print('Exporting to ONNX...')
export_path = best_model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True,   # ONNX simplifier — kurangi ukuran & percepat inference
    opset=17,
)

# Copy ONNX ke model/
import shutil
shutil.copy2(export_path, 'model/best.onnx')

# Check size
pt_size   = os.path.getsize('model/best.pt') / (1024**2)
onnx_size = os.path.getsize('model/best.onnx') / (1024**2)
print(f'\n=== MODEL SIZE CHECK ===')
print(f'PyTorch (.pt) : {pt_size:.2f} MB')
print(f'ONNX          : {onnx_size:.2f} MB')
print(f'Limit         : 50 MB')
assert onnx_size <= 50, f'CONSTRAINT VIOLATION: ONNX model {onnx_size:.2f} MB > 50 MB'
print(f'✅ Constraint C-A1 PASS: {onnx_size:.2f} MB ≤ 50 MB')

## 7. Training Results Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path('runs/train/rescuevision_v1')

# Plot training curves
results_png = results_dir / 'results.png'
if results_png.exists():
    img = mpimg.imread(str(results_png))
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves')
    plt.tight_layout()
    plt.show()

# PR curve
pr_png = results_dir / 'PR_curve.png'
if pr_png.exists():
    img = mpimg.imread(str(pr_png))
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PR Curve')
    plt.show()

## 8. Summary
Jalankan cell ini terakhir untuk summary yang bisa dikopi ke Proposal Bab 3.

In [ ]:
import time
import onnxruntime as ort
import numpy as np

# Quick CPU inference time check
session = ort.InferenceSession('model/best.onnx', providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)

# Warmup
for _ in range(3):
    session.run(None, {input_name: dummy})

# Benchmark 20 runs
times = []
for _ in range(20):
    t = time.perf_counter()
    session.run(None, {input_name: dummy})
    times.append(time.perf_counter() - t)

mean_ms = np.mean(times) * 1000
max_ms  = np.max(times)  * 1000

print('=' * 50)
print('CONSTRAINT COMPLIANCE SUMMARY (untuk Bab 3)')
print('=' * 50)
print(f'C-A1 Model Size    : {onnx_size:.2f} MB ≤ 50 MB   ✅')
print(f'C-A2 CPU Compat    : ONNX Runtime CPU-only         ✅')
print(f'C-A3 Inference Time: {mean_ms:.1f} ms mean, {max_ms:.1f} ms max ≤ 3000 ms ✅' if max_ms <= 3000 else f'C-A3 FAIL: {max_ms:.1f} ms > 3000 ms ❌')
print(f'C-A4 Framework     : Ultralytics YOLOv8 + ONNX     ✅')
print(f'C-A5 Offline Total : No external API               ✅')
print('=' * 50)